# 02 - Ryanair fare calendars

Fetches per-day fare lists per (origin, destination, direction) so the app can
let users pick a date range and see real prices.

- Per-day shape keeps time + flight number: `{date: [{time, price, flight}, ...]}`
  sorted cheapest-first.
- Resumable cache (`fare_calendars_monthly.json`) - re-runs pick up where they
  stopped. The 429 rate-limit backoff handles bursts.
- `MAX_RUN_MINUTES` bails out cleanly after N minutes with the cache flushed,
  for incremental overnight runs.
- The sanity-check cell at the end flags routes that look suspiciously empty.

## 1. Setup

In [ ]:
import sys, subprocess, json, time, random
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from collections import defaultdict

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
        return
    except ModuleNotFoundError:
        pass
    # Try a normal install; fall back to --user if pip refuses system installs
    for args in ([sys.executable, "-m", "pip", "install", "-q", pkg], 
                 [sys.executable, "-m", "pip", "install", "-q", "--user", pkg]):
        if subprocess.run(args).returncode == 0:
            __import__(name)
            return
    raise RuntimeError(f"Could not install {pkg}. Install it manually: pip install {pkg}")

_ensure("ryanair-py", "ryanair")
_ensure("tqdm")

from ryanair import Ryanair
from tqdm.auto import tqdm

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

cfg    = json.loads((CACHE_DIR / "config.json").read_text(encoding="utf-8"))
master = json.loads((CACHE_DIR / "destinations_master.json").read_text(encoding="utf-8"))

origins      = cfg["origins"]
start_date   = date.fromisoformat(cfg["start_date"])
end_date     = date.fromisoformat(cfg["end_date"])
destinations = master["destinations"]
airports     = master["airports"]

print(f"Origins: {origins}")
print(f"Window: {start_date} to {end_date}  ({(end_date - start_date).days} days)")
print(f"Destinations: {len(destinations)}")

## 2. Resolve each destination's Ryanair anchor airport

In [15]:
def resolve_anchor(d):
    if d["tier"] == "airport":
        if d.get("ryanair_serves"):
            return (d["iata"], 0, 0)
        return None
    for iata, mins, eur in d["nearest_airports"]:
        a = airports.get(iata)
        if a and a.get("ryanair_serves"):
            return (iata, mins, eur)
    return None

resolved = {}
no_route = []
for d in destinations:
    anchor = resolve_anchor(d)
    if anchor is None:
        no_route.append(d["id"])
    else:
        resolved[d["id"]] = anchor

print(f"Resolved: {len(resolved)} destinations have a Ryanair route")
print(f"No Ryanair route: {len(no_route)} destinations (will be flagged in app)")
if no_route[:10]:
    print(f"  Examples: {no_route[:10]}")

Resolved: 439 destinations have a Ryanair route
No Ryanair route: 11 destinations (will be flagged in app)
  Examples: ['BRN', 'LNZ', 'BOO', 'AES', 'KRN', 'VBY', 'INV', 'gem:lofoten', 'gem:geiranger', 'gem:kiruna']


## 3. Build the fetch queue

In [16]:
anchor_iatas = sorted({iata for iata, _, _ in resolved.values()})
print(f"Unique anchor airports: {len(anchor_iatas)}")

def month_ranges(d0, d1):
    cur = date(d0.year, d0.month, 1)
    while cur <= d1:
        if cur.month == 12:
            nxt = date(cur.year + 1, 1, 1)
        else:
            nxt = date(cur.year, cur.month + 1, 1)
        s = max(cur, d0)
        e = min(nxt - timedelta(days=1), d1)
        yield (cur.year, cur.month, s, e)
        cur = nxt

queue = []
for origin in origins:
    for dest_iata in anchor_iatas:
        if dest_iata == origin:
            continue
        for y, m, s, e in month_ranges(start_date, end_date):
            queue.append((origin, dest_iata, "out", y, m, s, e))
            queue.append((origin, dest_iata, "ret", y, m, s, e))

print(f"Queue size: {len(queue)} month-direction pairs")

Unique anchor airports: 257
Queue size: 4096 month-direction pairs


## 4. Load / init resumable cache

In [ ]:
MONTHLY_CACHE = CACHE_DIR / "fare_calendars_monthly.json"
CACHE_SCHEMA  = 7

def is_v6_entry(v):
    if not isinstance(v, dict):
        return False
    if "_error" in v:
        return True
    if not v:
        return True
    sample_val = next(iter(v.values()))
    return isinstance(sample_val, list)

def is_valid_key(k):
    parts = k.split("|")
    return len(parts) == 4 and len(parts[3]) == 7 and parts[3][4] == "-"

raw_cache = {}
if MONTHLY_CACHE.exists():
    try:
        raw_cache = json.loads(MONTHLY_CACHE.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        print("cache file unparseable, starting fresh")
        raw_cache = {}

cache = {}
dropped_cruft = 0
dropped_old   = 0
for k, v in raw_cache.items():
    if not is_valid_key(k):
        dropped_cruft += 1
        continue
    if not is_v6_entry(v):
        dropped_old += 1
        continue
    cache[k] = v

if dropped_cruft:
    print(f"  dropped {dropped_cruft} non-key entries (old schema)")
if dropped_old:
    print(f"  dropped {dropped_old} pre-v6 entries (will be re-fetched)")

def cache_key(origin, dest, direction, year, month):
    return f"{origin}|{dest}|{direction}|{year:04d}-{month:02d}"

to_fetch = [item for item in queue if cache_key(item[0], item[1], item[2], item[3], item[4]) not in cache]
print(f"Cache: {len(cache)} entries kept")
print(f"To fetch: {len(to_fetch)} (out of {len(queue)} total queue items)")

## 5. Fetch loop with tqdm progress

**Knobs:**
- `RUN_NETWORK` — must be `True` to actually fetch.
- `MAX_RUN_MINUTES` — cleanly exit after N minutes (cache flushed). `None` = no limit. Useful for incremental runs.
- `BASELINE_DELAY_S` — pacing between successful calls. 5s is the sweet spot; 429 backoff handles bursts.

**Progress bar** shows live counters and ETA, updating every iteration so you see immediate movement.

**Cache flushed** every 25 successful fetches AND on graceful exit (Ctrl+C, time limit, exception). You can't lose more than 25 fetches of progress.

In [ ]:
RUN_NETWORK      = True     # set to True to fetch
MAX_RUN_MINUTES  = None     # e.g. 60 for a one-hour run; None = no limit
BASELINE_DELAY_S = 5.0
JITTER_S         = 1.0
BACKOFFS         = [60, 120, 240]
FLUSH_EVERY      = 25

def _flush():
    MONTHLY_CACHE.write_text(json.dumps(cache, indent=2), encoding="utf-8")

if RUN_NETWORK and to_fetch:
    r = Ryanair(currency="EUR")
    consecutive_429s = 0
    stats = {"fetched": 0, "empty": 0, "err": 0}
    started = time.time()
    deadline = started + MAX_RUN_MINUTES * 60 if MAX_RUN_MINUTES else None
    stopped_reason = None

    bar = tqdm(to_fetch, desc="Ryanair", unit="call", smoothing=0.05)
    try:
        for (origin, dest, direction, y, m, s, e) in bar:
            if deadline and time.time() > deadline:
                stopped_reason = f"hit MAX_RUN_MINUTES={MAX_RUN_MINUTES}"
                break

            k = cache_key(origin, dest, direction, y, m)
            if k in cache:
                continue

            from_airport, to_airport = (origin, dest) if direction == "out" else (dest, origin)

            attempt = 0
            while True:
                try:
                    fares = r.get_cheapest_flights(
                        from_airport, s, e,
                        destination_airport=to_airport,
                    )
                    cal = defaultdict(list)
                    for f in fares or []:
                        dt = f.departureTime
                        cal[dt.date().isoformat()].append({
                            "time":   dt.strftime("%H:%M"),
                            "price":  round(float(f.price), 2),
                            "flight": getattr(f, "flightNumber", ""),
                        })
                    cleaned = {}
                    for d_iso, entries in cal.items():
                        seen = set()
                        deduped = []
                        for ent in sorted(entries, key=lambda x: x["price"]):
                            sig = (ent["time"], ent["flight"])
                            if sig in seen:
                                continue
                            seen.add(sig)
                            deduped.append(ent)
                        cleaned[d_iso] = deduped
                    cache[k] = cleaned
                    if cleaned:
                        stats["fetched"] += 1
                    else:
                        stats["empty"] += 1
                    consecutive_429s = 0
                    break
                except Exception as ex:
                    msg = str(ex).lower()
                    if "429" in msg or "rate" in msg:
                        consecutive_429s += 1
                        if attempt >= len(BACKOFFS) - 1 or consecutive_429s >= 3:
                            cache[k] = {"_error": "rate_limit"}
                            stats["err"] += 1
                            break
                        time.sleep(BACKOFFS[attempt])
                        attempt += 1
                    else:
                        cache[k] = {"_error": str(ex)[:200]}
                        stats["err"] += 1
                        break

            bar.set_postfix(
                fetched=stats["fetched"],
                empty=stats["empty"],
                err=stats["err"],
            )

            done = stats["fetched"] + stats["empty"] + stats["err"]
            if done and done % FLUSH_EVERY == 0:
                _flush()

            time.sleep(BASELINE_DELAY_S + random.uniform(-JITTER_S, JITTER_S))

            if consecutive_429s >= 3:
                stopped_reason = "3 consecutive 429s"
                break
    except KeyboardInterrupt:
        stopped_reason = "KeyboardInterrupt"
    finally:
        bar.close()
        _flush()

    elapsed = time.time() - started
    print(f"\nStopped: {stopped_reason or 'queue exhausted'}")
    print(f"  elapsed: {elapsed/60:.1f} min")
    print(f"  fetched (with fares): {stats['fetched']}")
    print(f"  empty (no fares):     {stats['empty']}")
    print(f"  errored:              {stats['err']}")
    print(f"  total cache entries:  {len(cache)}")
elif not RUN_NETWORK:
    print("RUN_NETWORK is False - set to True to fetch")
else:
    print("Nothing to fetch - cache already complete for current queue")

## 6. Aggregate into fare_calendars.json

In [ ]:
routes = defaultdict(dict)
n_errors = 0

for k, v in cache.items():
    if not isinstance(v, dict) or "_error" in v:
        n_errors += 1
        continue
    parts = k.split("|")
    if len(parts) != 4:
        continue
    origin, dest, direction, _ym = parts
    target = routes[(origin, dest, direction)]
    for d_iso, entries in v.items():
        if d_iso in target:
            combined = target[d_iso] + entries
            seen = set()
            merged = []
            for ent in sorted(combined, key=lambda x: x["price"]):
                sig = (ent["time"], ent["flight"])
                if sig in seen:
                    continue
                seen.add(sig)
                merged.append(ent)
            target[d_iso] = merged
        else:
            target[d_iso] = entries

fare_calendars = {}
for (origin, dest, direction), cal in routes.items():
    per_day = {}
    for d_iso, entries in cal.items():
        if not entries:
            continue
        per_day[d_iso] = {
            "cheapest": entries[0]["price"],
            "flights":  entries,
        }
    fare_calendars.setdefault(origin, {}).setdefault(dest, {})[direction] = per_day

OUT = CACHE_DIR / "fare_calendars.json"
payload = {
    "meta": {
        "generated_at":   datetime.now(timezone.utc).isoformat(),
        "schema_version": CACHE_SCHEMA,
        "n_routes":       sum(len(v) for v in fare_calendars.values()),
        "n_errors":       n_errors,
        "origins":        origins,
        "start_date":     start_date.isoformat(),
        "end_date":       end_date.isoformat(),
        "per_day_shape":  "{cheapest: float, flights: [{time, price, flight}]}",
    },
    "fare_calendars": fare_calendars,
}
OUT.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Wrote {OUT} - {payload['meta']['n_routes']} routes, {n_errors} errors")

## 7. Sanity check - are the empties suspicious?

If Ryanair was blocking your IP, calls would come back empty (the library swallows 403s after retry exhaustion). We can't tell from one empty - but if **all months of a known-real route are empty**, something's wrong.

This cell:
1. Reports overall fill rate per route.
2. Flags a handful of known-real Ryanair routes (BRU/CRL to MAD, BCN, BGY, DUB, STN). If any of those come back empty across 2+ months, you were almost certainly blocked.

**If known-real routes are empty**, you can purge them and retry from a different IP:
```python
cache = {k: v for k, v in cache.items() if v and "_error" not in v}
MONTHLY_CACHE.write_text(json.dumps(cache, indent=2), encoding="utf-8")
```
Then re-run cell 4 then 5.

In [ ]:
route_state = {}
for k, v in cache.items():
    if not is_valid_key(k):
        continue
    if not isinstance(v, dict) or "_error" in v:
        continue
    o, d, dr, _ = k.split("|")
    st = route_state.setdefault((o, d, dr), {"months": 0, "with_fares": 0})
    st["months"] += 1
    if v:
        st["with_fares"] += 1

total_routes        = len(route_state)
fully_empty_routes  = sum(1 for s in route_state.values() if s["with_fares"] == 0)
fully_filled_routes = sum(1 for s in route_state.values() if s["with_fares"] == s["months"] and s["months"] > 0)

print(f"Routes evaluated: {total_routes}")
print(f"  fully filled (data for every fetched month): {fully_filled_routes}")
print(f"  fully empty  (no fares in any fetched month): {fully_empty_routes}")
if total_routes:
    print(f"  fully-empty rate: {100*fully_empty_routes/total_routes:.1f}%")

KNOWN_REAL = [
    ("BRU", "MAD"), ("BRU", "BCN"), ("BRU", "VLC"), ("BRU", "AGP"),
    ("CRL", "MAD"), ("CRL", "BCN"), ("CRL", "BGY"), ("CRL", "DUB"),
    ("CRL", "STN"), ("CRL", "VLC"), ("CRL", "AGP"), ("CRL", "ALC"),
]
print("\nKnown-real route check (should NOT all be empty):")
suspicious = []
any_evaluated = False
for o, d in KNOWN_REAL:
    out = route_state.get((o, d, "out"))
    ret = route_state.get((o, d, "ret"))
    if not out and not ret:
        print(f"  {o}-{d}: not yet fetched")
        continue
    any_evaluated = True
    out_str = f"{out['with_fares']}/{out['months']}" if out else "-"
    ret_str = f"{ret['with_fares']}/{ret['months']}" if ret else "-"
    out_empty = out and out["with_fares"] == 0 and out["months"] >= 2
    ret_empty = ret and ret["with_fares"] == 0 and ret["months"] >= 2
    flag = "  SUSPICIOUS" if (out_empty or ret_empty) else ""
    if out_empty or ret_empty:
        suspicious.append((o, d))
    print(f"  {o}-{d}: out={out_str}, ret={ret_str}{flag}")

if suspicious:
    print(f"\n{len(suspicious)} known-real routes returned empty - likely blocked by Ryanair.")
    print("  Try a different IP (VPN / hotspot), or accept those routes are unfetchable for now.")
elif any_evaluated:
    print("\nKnown-real routes have data - fetch looks healthy.")

## 8. Date-range query helper

In [ ]:
def cheapest_round_trip(origin, dest, range_start, range_end, min_nights=1):
    """Cheapest valid out+ret combo in a date range. Returns None if no data."""
    try:
        out_cal = fare_calendars[origin][dest]["out"]
        ret_cal = fare_calendars[origin][dest]["ret"]
    except KeyError:
        return None

    out_days = sorted(d for d in out_cal if range_start <= d <= range_end)
    ret_days = sorted(d for d in ret_cal if range_start <= d <= range_end)

    best = None
    for od in out_days:
        out_entry = out_cal[od]
        od_dt = date.fromisoformat(od)
        earliest_ret = (od_dt + timedelta(days=min_nights)).isoformat()
        for rd in ret_days:
            if rd < earliest_ret:
                continue
            ret_entry = ret_cal[rd]
            total = out_entry["cheapest"] + ret_entry["cheapest"]
            if best is None or total < best["total"]:
                best = {
                    "total": round(total, 2),
                    "out": {"date": od, "time": out_entry["flights"][0]["time"],
                            "price": out_entry["cheapest"], "flight": out_entry["flights"][0]["flight"]},
                    "ret": {"date": rd, "time": ret_entry["flights"][0]["time"],
                            "price": ret_entry["cheapest"], "flight": ret_entry["flights"][0]["flight"]},
                }
    return best

if fare_calendars:
    demo = None
    for o, dests in fare_calendars.items():
        for d, dirs in dests.items():
            if dirs.get("out") and dirs.get("ret"):
                demo = cheapest_round_trip(o, d, start_date.isoformat(), end_date.isoformat(), min_nights=7)
                if demo:
                    print(f"Demo cheapest 7+ night trip {o}-{d}:")
                    print(json.dumps(demo, indent=2))
                    break
        if demo:
            break
    if not demo:
        print("No route has both out+ret data with valid combos yet.")
else:
    print("No fare data - run cell 5 with RUN_NETWORK=True.")